# Header Event Analysis

Evaluate one inference folder at a time against the positive labelled-header spreadsheets.

Change the parameters in the next cell and run the notebook from the repo root.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def find_header_net_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents, Path("/home/aerial/repository/heimdall_net/header_net")]
    for candidate in candidates:
        if (candidate / "SoccerNet").exists() and (candidate / "analysis" / "header_event_analysis.py").exists():
            return candidate
    raise RuntimeError(
        "Could not find the header_net repo root. Open VS Code on the header_net folder, or set BASE_DIR manually."
    )

BASE_DIR = find_header_net_root()
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from analysis.header_event_analysis import (
    compute_event_metrics_curve,
    evaluate_event_detection,
    filter_rows_by_frame_windows,
    load_inference_folder,
    load_labelled_headers,
    plot_event_metrics_vs_threshold,
    plot_frame_roc_curve,
    plot_match_timeline,
    summarize_event_metrics_by_video,
)

INFERENCE_FOLDER = BASE_DIR / "output" / "2026_03_31_ball-frame_vmae2-base_inference_result" / "test_inference"
EPOCH_FOLDERS = {
    "latest": BASE_DIR / "output" / "2026_03_31_ball-frame_vmae2-base_inference_result" / "test_inference",
    "ep020": BASE_DIR / "output" / "2026_03_31_ball-frame_vmae2-base_inference_result" / "test_inference_ep-020",
    "ep025": BASE_DIR / "output" / "2026_03_31_ball-frame_vmae2-base_inference_result" / "test_inference_ep-025",
}
LABEL_DIR = BASE_DIR / "SoccerNet" / "test" / "labelled_header"
FRAME_FILTERS = {
    ("2015-11-25-22-45BMonchengladbach4-2Sevilla", 1): (31396, 166760),
    ("2015-11-25-22-45BMonchengladbach4-2Sevilla", 2): (39875, 181670),
}

THRESHOLD = 0.50
MERGE_GAP = 3  # change this when you want to merge positive bursts more or less aggressively
TOL_LEFT = 7
TOL_RIGHT = 8

MATCH_ID = "2015-11-25-22-45BMonchengladbach4-2Sevilla"
HALF = 1
FRAME_WINDOW = None  # example: (50000, 70000)
THRESHOLD_GRID = np.linspace(0.0, 1.0, 101)

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

In [ ]:
inference_df = filter_rows_by_frame_windows(
    load_inference_folder(INFERENCE_FOLDER, repo_root=BASE_DIR),
    FRAME_FILTERS,
    frame_column="frame",
)
gt_df = filter_rows_by_frame_windows(
    load_labelled_headers(LABEL_DIR),
    FRAME_FILTERS,
    frame_column="gt_frame",
)

result = evaluate_event_detection(
    inference_df,
    gt_df,
    threshold=THRESHOLD,
    merge_gap=MERGE_GAP,
    tol_left=TOL_LEFT,
    tol_right=TOL_RIGHT,
)

overall_metrics = pd.DataFrame([result.metrics])
per_video_metrics = summarize_event_metrics_by_video(result)

print("Inference source:", inference_df.attrs.get("per_match_dir"))
print("Label source:", gt_df.attrs.get("label_dir"))
print("Frame filters:", FRAME_FILTERS)
display(overall_metrics)
display(per_video_metrics)

In [ ]:
selected_gt = result.gt_events[
    (result.gt_events["video_id"] == MATCH_ID) & (result.gt_events["half"] == HALF)
][["gt_event_id", "gt_frame", "match_status", "matched_pred_event_id"]]

selected_pred = result.pred_events[
    (result.pred_events["video_id"] == MATCH_ID) & (result.pred_events["half"] == HALF)
][[
    "pred_event_id",
    "start_frame",
    "end_frame",
    "peak_frame",
    "peak_prob",
    "n_positive_frames",
    "match_status",
    "matched_gt_event_id",
]]

selected_matches = result.matches[
    (result.matches["video_id"] == MATCH_ID) & (result.matches["half"] == HALF)
]

display(selected_gt)
display(selected_pred)
display(selected_matches)

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))
plot_match_timeline(
    inference_df,
    result,
    match_id=MATCH_ID,
    half=HALF,
    threshold=THRESHOLD,
    merge_gap=MERGE_GAP,
    tol_left=TOL_LEFT,
    tol_right=TOL_RIGHT,
    frame_window=FRAME_WINDOW,
    ax=ax,
)
plt.show()

In [ ]:
curve_df = compute_event_metrics_curve(
    inference_df,
    gt_df,
    thresholds=THRESHOLD_GRID,
    merge_gap=MERGE_GAP,
    tol_left=TOL_LEFT,
    tol_right=TOL_RIGHT,
)

fig, ax = plt.subplots(figsize=(10, 5))
plot_event_metrics_vs_threshold(curve_df, current_threshold=THRESHOLD, ax=ax)
plt.show()

display(curve_df.head())
display(curve_df.tail())

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
roc_auc = plot_frame_roc_curve(inference_df, ax=ax)
plt.show()
print(f"Frame-level ROC AUC: {roc_auc:.4f}")